In [55]:
import os
import pandas as pd
from dotenv import load_dotenv
#.env の読み込み
load_dotenv(override=True)
#.env からエクセルのパスを取得
excel_path = os.getenv('COFFEE_SALES_EXCEL_PATH')
#エクセルファイルの読み込み
transaction_df = pd.read_excel(excel_path, sheet_name='Transactions')
transaction_df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,Revenue,Month,Month.1,Weekday,Weekday.1,Hour
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0,1,Jan,7,Sun,7
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,1,Jan,7,Sun,7
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0,1,Jan,7,Sun,7
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0,1,Jan,7,Sun,7
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,1,Jan,7,Sun,7


In [56]:
transaction_df.isnull().sum()

transaction_id      0
transaction_date    0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
Revenue             0
Month               0
Month.1             0
Weekday             0
Weekday.1           0
Hour                0
dtype: int64

In [57]:
#transactiont_dateをdatetime型に変換し、期間を確認する
transaction_df['transaction_date'] = pd.to_datetime(transaction_df['transaction_date'])
print(transaction_df['transaction_date'].min())
print(transaction_df['transaction_date'].max())

2023-01-01 00:00:00
2023-06-30 00:00:00


In [58]:
#unit_priceとproduct_categoryで料金表を作る
price_df = transaction_df[['unit_price', 'product_category']].drop_duplicates().reset_index(drop=True)
price_df.head()

,unit_price,product_category
0,3.0,Coffee
1,3.1,Tea
2,4.5,Drinking Chocolate
3,2.0,Coffee
4,3.0,Bakery


In [59]:
#unit?priveでevenuを割って販売個数を作る
transaction_df['quantity'] = (transaction_df['Revenue'] / transaction_df['unit_price']).round().astype(int)
transaction_df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,Revenue,Month,Month.1,Weekday,Weekday.1,Hour,quantity
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0,1,Jan,7,Sun,7,2
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,1,Jan,7,Sun,7,2
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0,1,Jan,7,Sun,7,2
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0,1,Jan,7,Sun,7,1
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,1,Jan,7,Sun,7,2


In [65]:
#quantityが2以上の場合は、2名以上で来店していると考えられる。2名、3名、4名以上で条件を作るため、group_sizeを作る
#Bakeryは、飲み物と一緒に購入と考えられるので、集計から除外することで、より正確な来店人数を把握できると考えられる
excluded_d = transaction_df[~transaction_df['product_category'].str.contains('Bakery')]
excluded_df['group_size'] = excluded_df['quantity'].apply(lambda x: '4 or more' if x >= 4 else '3' if x >= 3 else '2' if x >= 2 else '1')
print(excluded_df.groupby('group_size')['quantity'].count())

group_size
1            64755
2            58276
3             3253
4 or more       36
Name: quantity, dtype: int64


C:\Users\user\AppData\Local\Temp\ipykernel_17404\1587711391.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  excluded_df['group_size'] = excluded_df['quantity'].apply(lambda x: '4 or more' if x >= 4 else '3' if x >= 3 else '2' if x >= 2 else '1')


In [64]:
#store_locationを使って何店舗がデータに含まているか確認する
print(excluded_df['store_location'].unique())
#先ほどの来店人数と店舗数を組み合わせて、店舗ごとの来店人数の分布を確認する
print(excluded_df.groupby(['store_location', 'group_size'])['quantity'].count())
#Hourを使って、午前、昼間、夕方、夜の時間帯に分けて、時間帯ごとの来店人数の分布を確認する
excluded_df['Hour_group'] = excluded_df['Hour'].apply(lambda x: 'morning' if x < 12 else 'afternoon' if x < 16 else 'evening' )
print(excluded_df.groupby(['Hour_group', 'group_size'])['quantity'].count())


['Lower Manhattan' "Hell's Kitchen" 'Astoria']
store_location   group_size
Astoria          1             23125
                 2             20185
Hell's Kitchen   1             22237
                 2             20871
                 4 or more        10
Lower Manhattan  1             19393
                 2             17220
                 3              3253
                 4 or more        26
Name: quantity, dtype: int64
Hour_group  group_size
afternoon   1             15473
            2             14237
            3               619
            4 or more         5
evening     1             14247
            2             12925
            3               413
            4 or more         7
morning     1             35035
            2             31114
            3              2221
            4 or more        24
Name: quantity, dtype: int64


C:\Users\user\AppData\Local\Temp\ipykernel_17404\2636645326.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  excluded_df['Hour_group'] = excluded_df['Hour'].apply(lambda x: 'morning' if x < 12 else 'afternoon' if x < 16 else 'evening' )


In [62]:
#bakeryを除外する前にデータで人気の商品を確認する
print(transaction_df.groupby('product_category')['quantity'].sum().sort_values(ascending=False))

product_category
Coffee                89250
Tea                   69737
Bakery                23214
Drinking Chocolate    17457
Flavours              10511
Coffee beans           1828
Loose Tea              1210
Branded                 776
Packaged Chocolate      487
Name: quantity, dtype: int64


In [67]:
#時間帯、店舗による商品売り上げの違いを確認する
print(excluded_df.groupby(['product_category', 'Hour_group', 'store_location'])['quantity'].sum())

product_category  Hour_group  store_location 
Branded           afternoon   Astoria               57
                              Hell's Kitchen        16
                              Lower Manhattan       82
                  evening     Astoria               62
                              Hell's Kitchen        20
                                                 ...  
Tea               evening     Hell's Kitchen      5366
                              Lower Manhattan     3174
                  morning     Astoria            10318
                              Hell's Kitchen     12721
                              Lower Manhattan    14114
Name: quantity, Length: 72, dtype: int64
